# Compression-01b — Kraft et McMillan en Lean 4 : le carnet compagnon du lake

Ce carnet est le **compagnon exécutable** du lake `Compression_lean` (EPIC #18706, décision D4) : il importe les modules `PrefixCodes`, `Decode` et `McMillan` **réels** du lake et les fait travailler — aucune définition n'est recopiée, ce que vous voyez exécuter est le livrable formel lui-même.

**Position dans la série** — le lake est le jumeau formel des carnets Python :
- `Compression-01-ShannonFano-Prefixe-Python.ipynb` (pli 1) : codes préfixes, Shannon–Fano, **inégalité de Kraft** → ancré par `PrefixCodes` ;
- `Compression-02-HuffmanCanonique-Python.ipynb` (pli 2, en livraison) : tables de décodage → ancré par `Decode` ;
- le **théorème de McMillan** (réciproque de Kraft) → ancré par `McMillan` (pli 3).

**Convention d'exécution** : carnet exécuté depuis la racine du lake (le kernel `lean4-wsl` détecte le `lakefile.lean` et résout les imports via `lake env`). C'est pourquoi ce fichier vit dans `Compression_lean/` : son répertoire EST son environnement.

## 0. Charger le lake

Les trois modules racine du lake : `PrefixCodes` (codes préfixes, Kraft), `Decode` (décodage, round-trip), `McMillan` (réciproque de Kraft). Le namespace commun est `CompressionLean`.

In [1]:
import PrefixCodes
import Decode
import McMillan

open CompressionLean

#check kraft_le_one
#check decode_encode
#check mcmillan_converse

import PrefixCodes
import Decode
import McMillan

open CompressionLean

#check kraft_le_one
──────▶  CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1
#check decode_encode
──────▶  CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)
  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws
#check mcmillan_converse
──────▶  CompressionLean.mcmillan_converse (L : ℕ) (ls : List ℕ) (hsort : SortedLe ls) (hL : ∀ l ∈ ls, l ≤ L)
  (hmass : mass L ls ≤ 2 ^ L) : ∃ ws, List.Pairwise PrefixFreeR ws ∧ List.map List.length ws = ls
--% env 0

Raw input:
{"cmd": "import PrefixCodes\nimport Decode\nimport McMillan\n\nopen CompressionLean\n\n#check kraft_le_one\n#check decode_encode\n#check mcmillan_converse"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)\n  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "CompressionLean.mcmillan_converse (L : ℕ) (ls : List ℕ) (hsort : SortedLe ls) (hL : ∀ l ∈ ls, l ≤ L)\n  (hmass : mass L ls ≤ 2 ^ L) : ∃ ws, List.Pairwise PrefixFreeR ws ∧ List.map List.length ws = ls"}],
 "env": 0}

Les trois résultats centraux, lus comme des contrats :

- `kraft_le_one` : la somme de Kraft de **tout** arbre de code est ≤ 1 — aucune implosion possible ;
- `decode_encode` : décoder la concaténation de mots de feuilles redonne **exactement** la liste émise — le contrat d'une table de décodage ;
- `mcmillan_converse` : si une bande de longueurs vérifie Kraft (masse ≤ 2^L), alors un code préfixe avec **exactement ces longueurs** existe — la réciproque, constructive.

## 1. Codes préfixes et somme de Kraft

Pour un arbre de code binaire `t`, `leafWords t` est la liste des mots des feuilles et `kraftSum t` la somme Σ 2^(-|w|). Le lake fournit l'arbre d'exemple `sf4` — le code Shannon–Fano à 4 symboles du carnet 01 (longueurs 1, 2, 3, 3).

In [2]:
-- Le code sf4 : ses mots de feuilles et sa somme de Kraft
#eval leafWords sf4
#eval kraftSum sf4
#eval (leafWords sf4).map List.length

-- Le code sf4 : ses mots de feuilles et sa somme de Kraft
#eval leafWords sf4
─────▶  [[true], [false, true], [false, false, true], [false, false, false]]
#eval kraftSum sf4
─────▶  1
#eval (leafWords sf4).map List.length
─────▶  [1, 2, 3, 3]
--% env 1

Raw input:
{"cmd": "-- Le code sf4 : ses mots de feuilles et sa somme de Kraft\n#eval leafWords sf4\n#eval kraftSum sf4\n#eval (leafWords sf4).map List.length", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[[true], [false, true], [false, false, true], [false, false, false]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "[1, 2, 3, 3]"}],
 "env": 1}

In [3]:
-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,
-- Kraft sature exactement a 1.
#eval kraftSum (full 3)
#eval (leafWords (full 3)).length
#check kraft_le_one
#check kraft_full

-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,
-- Kraft sature exactement a 1.
#eval kraftSum (full 3)
─────▶  1
#eval (leafWords (full 3)).length
─────▶  8
#check kraft_le_one
──────▶  CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1
#check kraft_full
──────▶  CompressionLean.kraft_full (k : ℕ) : kraftSum (full k) = 1
--% env 2

Raw input:
{"cmd": "-- Un arbre complet de profondeur k : 2^k feuilles, toutes a la meme profondeur,\n-- Kraft sature exactement a 1.\n#eval kraftSum (full 3)\n#eval (leafWords (full 3)).length\n#check kraft_le_one\n#check kraft_full", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "8"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "CompressionLean.kraft_le_one (t : CodeTree) : kraftSum t ≤ 1"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "CompressionLean.kraft_full (k : ℕ) : kraftSum (full k) = 1"}],
 "env": 2}

**Lecture** — `sf4` (longueurs 1, 2, 3, 3) sature aussi Kraft à 1 : chaque bit de la bande est consommé. `full 3` est l'arbre complet à 8 feuilles : Kraft = 1 = 2^3/2^3. Le théorème `kraft_le_one` garantit qu'aucun arbre ne dépasse 1 — c'est l'argument de non-contradiction du carnet 01.

### Exercice 1 — Prédire avant d'exécuter

Un arbre complet de profondeur 5 a combien de mots de feuilles, et quelle somme de Kraft ? Écrivez vos deux prédictions sur papier, puis exécutez la cellule.

In [4]:
-- Exercice 1 : verifiez vos predictions
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval (leafWords (full 5)).length
#eval kraftSum (full 5)

-- Exercice 1 : verifiez vos predictions
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval (leafWords (full 5)).length
─────▶  32
#eval kraftSum (full 5)
─────▶  1
--% env 3

Raw input:
{"cmd": "-- Exercice 1 : verifiez vos predictions\n-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule\n#eval (leafWords (full 5)).length\n#eval kraftSum (full 5)", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "32"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "1"}],
 "env": 3}

## 2. Décodage préfixe et round-trip

`decode1` lit **un** mot le long de l'arbre ; `decode` itère sur le flux complet. Les théorèmes `decode1_sound` / `decode1_complete` établissent la correction du pas, et `decode_encode` ferme le contrat : émettre la concaténation des mots puis décoder redonne la liste initiale, dans l'ordre, sans reste.

Le message du carnet 01 : `a b c` se code `0·10·110` (ici `[true, false, true, false, false, true]` en convention bit=Bool).

In [5]:
-- Decoder le message a b c du carnet 01
#eval decode sf4 [true, false, true, false, false, true]
-- et le message complet a b c d
#eval decode sf4 [true, false, true, false, false, true, false, false, false]

-- Decoder le message a b c du carnet 01
#eval decode sf4 [true, false, true, false, false, true]
─────▶  some [[true], [false, true], [false, false, true]]
-- et le message complet a b c d
#eval decode sf4 [true, false, true, false, false, true, false, false, false]
─────▶  some [[true], [false, true], [false, false, true], [false, false, false]]
--% env 4

Raw input:
{"cmd": "-- Decoder le message a b c du carnet 01\n#eval decode sf4 [true, false, true, false, false, true]\n-- et le message complet a b c d\n#eval decode sf4 [true, false, true, false, false, true, false, false, false]", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "some [[true], [false, true], [false, false, true]]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data":
   "some [[true], [false, true], [false, false, true], [false, false, false]]"}],
 "env": 4}

In [6]:
-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder
#eval decode sf4 (leafWords sf4).flatten
#check decode_encode

-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder
#eval decode sf4 (leafWords sf4).flatten
─────▶  some [[true], [false, true], [false, false, true], [false, false, false]]
#check decode_encode
──────▶  CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)
  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws
--% env 5

Raw input:
{"cmd": "-- Le round-trip sur TOUTE la table : concatener les mots de feuilles puis decoder\n#eval decode sf4 (leafWords sf4).flatten\n#check decode_encode", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "some [[true], [false, true], [false, false, true], [false, false, false]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "CompressionLean.decode_encode (t : CodeTree) (ws : List (List Bool)) (ht : t ≠ CodeTree.leaf)\n  (h : ∀ w ∈ ws, w ∈ leafWords t) : decode t ws.flatten = some ws"}],
 "env": 5}

**Lecture** — le round-trip rend `some (leafWords sf4)` : la table est complète, l'ordre est préservé.

### Exercice 2 — Un flux tronqué

Le flux `[false]` ne contient aucun mot complet (le seul mot de longueur 1 est `a` = `[true]`). Que répond `decode` ? Prédisez, puis exécutez. Que répond-il pour `[false, true]` ?

In [7]:
-- Exercice 2 : flux tronque puis complet
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval decode sf4 [false]
#eval decode sf4 [false, true]

-- Exercice 2 : flux tronque puis complet
-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule
#eval decode sf4 [false]
─────▶  none
#eval decode sf4 [false, true]
─────▶  some [[false, true]]
--% env 6

Raw input:
{"cmd": "-- Exercice 2 : flux tronque puis complet\n-- TODO etudiant : notez vos predictions AVANT d'executer cette cellule\n#eval decode sf4 [false]\n#eval decode sf4 [false, true]", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "none"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "some [[false, true]]"}],
 "env": 6}

## 3. McMillan : la réciproque de Kraft, constructivement

Kraft dit « tout code préfixe satisfait Σ 2^(-ℓᵢ) ≤ 1 ». McMillan renverse l'énoncé : **toute** bande de longueurs triée satisfaisant l'inégalité est réalisée par un code préfixe. La preuve du lake est constructive : `canon ls c` construit le code canonique (Schwartz–Miller 1964) par sauts de masse, `mass` mesure la bande à l'échelle 2^L (entiers, pas de rationnels), et `canon_spec` établit les deux moitiés : préfixe-liberté **et** longueurs exactes.

In [8]:
-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee
#eval canon [1, 2, 3, 3] 0
#eval mass 3 [1, 2, 3, 3]
-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111

-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee
#eval canon [1, 2, 3, 3] 0
─────▶  [[false], [true, false], [true, true, false], [true, true, true]]
#eval mass 3 [1, 2, 3, 3]
─────▶  8
-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111
--% env 7

Raw input:
{"cmd": "-- La bande du carnet 02 : longueurs (1, 2, 3, 3), masse saturee\n#eval canon [1, 2, 3, 3] 0\n#eval mass 3 [1, 2, 3, 3]\n-- 8 = 2^3 : Kraft sature, le code canonique retrouve a=0, b=10, c=110, d=111", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[[false], [true, false], [true, true, false], [true, true, true]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "8"}],
 "env": 7}

In [9]:
-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou
#eval canon [1, 3, 3] 0
#eval mass 3 [1, 3, 3]
-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer
#eval mass 3 [1, 1, 1]

-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou
#eval canon [1, 3, 3] 0
─────▶  [[false], [true, false, false], [true, false, true]]
#eval mass 3 [1, 3, 3]
─────▶  6
-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer
#eval mass 3 [1, 1, 1]
─────▶  12
--% env 8

Raw input:
{"cmd": "-- Une bande NON saturee : (1, 3, 3), masse 6 < 8 -- realisable malgre le trou\n#eval canon [1, 3, 3] 0\n#eval mass 3 [1, 3, 3]\n-- Une bande IMPOSSIBLE : (1, 1, 1), masse 12 > 8 -- aucun code prefixe ne peut l'habituer\n#eval mass 3 [1, 1, 1]", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[[false], [true, false, false], [true, false, true]]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "12"}],
 "env": 8}

In [10]:
-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms mcmillan_converse
#print axioms canon_spec
#print axioms decode_encode
#print axioms kraft_le_one

-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native
#print axioms mcmillan_converse
──────▶  'CompressionLean.mcmillan_converse' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms canon_spec
──────▶  'CompressionLean.canon_spec' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms decode_encode
──────▶  'CompressionLean.decode_encode' depends on axioms: [propext, Quot.sound]
#print axioms kraft_le_one
──────▶  'CompressionLean.kraft_le_one' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 9

Raw input:
{"cmd": "-- Les theoremes, et leurs axiomes : aucun sorry, aucune reduction native\n#print axioms mcmillan_converse\n#print axioms canon_spec\n#print axioms decode_encode\n#print axioms kraft_le_one", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'CompressionLean.mcmillan_converse' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'CompressionLean.canon_spec' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'CompressionLean.decode_encode' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'CompressionLean.kraft_le_one' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 9}

**Lecture** — les axiomes, pilier par pilier, tels que les imprime la cellule précédente : `mcmillan_converse`, `canon_spec` et `kraft_le_one` dépendent de `propext`, `Classical.choice` et `Quot.sound` ; `decode_encode` ne dépend que de `propext` et `Quot.sound` — c'est le seul des quatre qui n'utilise pas le choix classique. Aucune preuve ne contient `sorryAx`, et aucune ne passe par le noyau natif (`native_decide`). Le livrable formel est complet.

### Encart Mathlib — ce que l'organe fournit, ce que le lake ajoute

Mathlib porte déjà l'inégalité, au pin `db584cd6` (Mathlib 4.33) : module `Mathlib.InformationTheory.Coding.KraftMcMillan`, théorème `InformationTheory.kraft_mcmillan_inequality` — pour un code **uniquement décodable** `S` sur un alphabet de taille `D`, `∑ w ∈ S, (1 / D) ^ w.length ≤ 1` sur `ℝ`. L'organe existe donc : le lake ne le réimplante pas.

Ce que le lake ajoute est la **direction constructive** : `mcmillan_converse` établit que toute bande de longueurs triée, majorée par `L` et de masse `≤ 2^L` — l'hypothèse de Kraft `Σ 2^(-ℓᵢ) ≤ 1` multipliée par `2^L` — est **réalisable** par un code préfixe de ces longueurs exactes, et `canon_spec` que la construction canonique `canon` en produit un. S'y ajoute une formalisation sur `CodeTree` (`kraft_le_one`), sur les rationnels exacts, qui sert au décodage de la série. Au pin, `Mathlib/` ne contient aucune occurrence de `PrefixFree` ni de `Huffman` (recherche sur l'arbre complet) : l'encart est un **complément déclaré** à l'organe, pas une absence.

### Exercice 3 — Réaliser une bande libre

Choisissez une bande de longueurs triée, de masse **strictement inférieure** à 2^L (par exemple `[1, 3]` avec L = 3). Vérifiez : (a) sa masse, (b) que `canon` la réalise avec des longueurs exactes. Que remarquez-vous pour les mots produits ?

In [11]:
-- Exercice 3 : bande libre (1, 3)
-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez
#eval mass 3 [1, 3]
#eval canon [1, 3] 0
#eval (canon [1, 3] 0).map List.length

-- Exercice 3 : bande libre (1, 3)
-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez
#eval mass 3 [1, 3]
─────▶  5
#eval canon [1, 3] 0
─────▶  [[false], [true, false, false]]
#eval (canon [1, 3] 0).map List.length
─────▶  [1, 3]
--% env 10

Raw input:
{"cmd": "-- Exercice 3 : bande libre (1, 3)\n-- TODO etudiant : remplacez la bande ci-dessous par la votre, predisez, executez\n#eval mass 3 [1, 3]\n#eval canon [1, 3] 0\n#eval (canon [1, 3] 0).map List.length", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "5"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "[[false], [true, false, false]]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[1, 3]"}],
 "env": 10}

## Conclusion

Ce carnet a exécuté le lake, pas une copie : l'inégalité de Kraft (`kraft_le_one`), le round-trip de décodage (`decode_encode`) et la réciproque de McMillan (`mcmillan_converse`) sont les mêmes objets que ceux vérifiés par `lake build` et le gate `proof-integrity` en CI. Les ancres numériques rejoignent celles des carnets Python frères : le code (1, 2, 3, 3) du carnet 01/02 apparaît ici comme `sf4` (Kraft saturé), son message `a b c` se décode sans reste, et `canon` le reconstruit mot pour mot depuis la seule bande de longueurs.

**Suite de la série** — le pli 4 (trie canonique de Huffman) étendra le lake ; ce carnet s'enrichira de sa section correspondante.